In [1]:
"""
PlantVillage Plant Disease Classification - PRODUCTION READY
- No Lambda layer issues
- No serialization errors
- Proper preprocessing
- Built-in validation
"""

import os
import json
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from PIL import Image

print("="*70)
print(" PLANTVILLAGE DISEASE CLASSIFICATION - FINAL VERSION")
print("="*70)

2025-11-14 12:35:23.413266: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-11-14 12:35:23.592304: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-11-14 12:35:25.605389: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


 PLANTVILLAGE DISEASE CLASSIFICATION - FINAL VERSION


In [2]:
# ==================== CLEANUP ====================
print("\n🧹 Cleaning up old files...")
tf.keras.backend.clear_session()
for f in ['plant_disease_model.keras', 'best_model.keras', 'class_names.json']:
    if os.path.exists(f):
        os.remove(f)
        print(f"   Deleted: {f}")


🧹 Cleaning up old files...
   Deleted: best_model.keras
   Deleted: class_names.json


In [3]:
# ==================== GPU SETUP ====================
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"\n✅ GPU configured: {len(gpus)} device(s)")
    except RuntimeError as e:
        print(f"\n⚠️  GPU warning: {e}")
else:
    print("\n⚠️  No GPU detected - training will be slow")


✅ GPU configured: 1 device(s)


In [4]:
# ==================== CONFIGURATION ====================
BASE = "/mnt/c/datasets/plantvillage_split"
TRAIN_DIR = os.path.join(BASE, "train")
VAL_DIR = os.path.join(BASE, "val")
TEST_DIR = os.path.join(BASE, "test")

IMG_SIZE = (224, 224)
BATCH_SIZE = 16
WARMUP_EPOCHS = 10
FINETUNE_EPOCHS = 20
SEED = 42

tf.random.set_seed(SEED)
np.random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

print(f"\n📊 Configuration:")
print(f"   Image size: {IMG_SIZE}")
print(f"   Batch size: {BATCH_SIZE}")
print(f"   Warmup epochs: {WARMUP_EPOCHS}")
print(f"   Fine-tune epochs: {FINETUNE_EPOCHS}")


📊 Configuration:
   Image size: (224, 224)
   Batch size: 16
   Warmup epochs: 10
   Fine-tune epochs: 20


In [5]:
# ==================== LOAD CLASS NAMES ====================
print(f"\n📁 Loading class names from: {TRAIN_DIR}")
class_names = sorted([d for d in os.listdir(TRAIN_DIR) 
                     if os.path.isdir(os.path.join(TRAIN_DIR, d))])
num_classes = len(class_names)

print(f"✅ Found {num_classes} classes")
print(f"   First class: {class_names[0]}")
print(f"   Last class: {class_names[-1]}")

# Save class names
with open("class_names.json", "w") as f:
    json.dump(class_names, f, indent=2)
print(f"✅ Saved class_names.json")

# Verify specific classes exist
corn_idx = class_names.index("Corn_(maize)___healthy")
print(f"\n🔍 Index verification:")
print(f"   Corn_(maize)___healthy at index: {corn_idx}")


📁 Loading class names from: /mnt/c/datasets/plantvillage_split/train
✅ Found 38 classes
   First class: Apple___Apple_scab
   Last class: Tomato___healthy
✅ Saved class_names.json

🔍 Index verification:
   Corn_(maize)___healthy at index: 10


In [6]:
# ==================== LOAD DATASETS ====================
print(f"\n📁 Loading datasets...")

train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=SEED,
    label_mode='int'
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    VAL_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False,
    label_mode='int'
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=False,
    label_mode='int'
)

print(f"✅ Datasets loaded")
print(f"   Train batches: {len(train_ds)}")
print(f"   Val batches: {len(val_ds)}")
print(f"   Test batches: {len(test_ds)}")

# CRITICAL: Verify class order matches
if class_names != train_ds.class_names:
    print("\n❌ ERROR: Class order mismatch!")
    print(f"   JSON: {class_names[:5]}")
    print(f"   Dataset: {train_ds.class_names[:5]}")
    exit(1)
else:
    print(f"✅ Class order verified - matches dataset")


📁 Loading datasets...
Found 37997 files belonging to 38 classes.


I0000 00:00:1763123770.228771   14964 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 5561 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4060 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9


Found 8129 files belonging to 38 classes.
Found 8179 files belonging to 38 classes.
✅ Datasets loaded
   Train batches: 2375
   Val batches: 509
   Test batches: 512
✅ Class order verified - matches dataset


In [7]:

# ==================== CLASS WEIGHTS ====================
print(f"\n⚖️  Computing class weights...")
counts = {}
for cls in class_names:
    cls_path = os.path.join(TRAIN_DIR, cls)
    count = len([f for f in os.listdir(cls_path) 
                if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
    counts[cls] = count

total_samples = sum(counts.values())
class_weight = {i: total_samples / (num_classes * counts[cls]) 
                for i, cls in enumerate(class_names)}

print(f"✅ Class weights computed")
print(f"   Total training samples: {total_samples}")


⚖️  Computing class weights...
✅ Class weights computed
   Total training samples: 37997


In [8]:
# ==================== OPTIMIZE PIPELINE ====================
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)
test_ds = test_ds.prefetch(buffer_size=AUTOTUNE)


In [9]:
# ==================== BUILD MODEL ====================
print(f"\n🏗️  Building model...")

# Load MobileNetV2 base
base_model = tf.keras.applications.MobileNetV2(
    input_shape=IMG_SIZE + (3,),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False

# Build complete model WITHOUT Lambda layer
inputs = layers.Input(shape=IMG_SIZE + (3,), name='input')

# Apply preprocessing directly (no Lambda)
x = inputs
# MobileNetV2 preprocessing: scale to [-1, 1]
x = layers.Rescaling(scale=1./127.5, offset=-1)(x)

# Base model
x = base_model(x, training=False)
# Classification head
x = layers.GlobalAveragePooling2D(name='gap')(x)
x = layers.Dropout(0.3, name='dropout')(x)
outputs = layers.Dense(num_classes, activation='softmax', name='predictions')(x)

# Create model
model = models.Model(inputs, outputs, name='PlantVillage_MobileNetV2')

print(f"✅ Model built successfully")
print(f"   Total params: {model.count_params():,}")
print(f"   Input shape: {model.input_shape}")
print(f"   Output shape: {model.output_shape}")

# Verify output size
if model.output_shape[-1] != num_classes:
    print(f"\n❌ ERROR: Output mismatch!")
    print(f"   Expected: {num_classes}, Got: {model.output_shape[-1]}")
    exit(1)



🏗️  Building model...
✅ Model built successfully
   Total params: 2,306,662
   Input shape: (None, 224, 224, 3)
   Output shape: (None, 38)


In [10]:
# ==================== CALLBACKS ====================
checkpoint = callbacks.ModelCheckpoint(
    filepath='best_model.keras',
    monitor='val_accuracy',
    mode='max',
    save_best_only=True,
    verbose=1
)

reduce_lr = callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    min_lr=1e-7,
    verbose=1
)

early_stopping = callbacks.EarlyStopping(
    monitor='val_loss',
    patience=7,
    restore_best_weights=True,
    verbose=1
)

In [11]:
# ==================== PHASE 1: WARMUP ====================
print(f"\n{'='*70}")
print(f" PHASE 1: WARMUP TRAINING ({WARMUP_EPOCHS} epochs)")
print(f"{'='*70}")

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("\n🚀 Starting warmup training...")
history_warmup = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=WARMUP_EPOCHS,
    callbacks=[checkpoint, reduce_lr],
    class_weight=class_weight,
    verbose=1
)
best_warmup_acc = max(history_warmup.history['val_accuracy'])
print(f"\n✅ Warmup complete!")
print(f"   Best validation accuracy: {best_warmup_acc*100:.2f}%")


 PHASE 1: WARMUP TRAINING (10 epochs)

🚀 Starting warmup training...
Epoch 1/10


2025-11-14 12:36:35.514327: I external/local_xla/xla/service/service.cc:163] XLA service 0x700a44003d20 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2025-11-14 12:36:35.514354: I external/local_xla/xla/service/service.cc:171]   StreamExecutor device (0): NVIDIA GeForce RTX 4060 Laptop GPU, Compute Capability 8.9
2025-11-14 12:36:35.598951: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2025-11-14 12:36:36.094742: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 91400
2025-11-14 12:36:42.090081: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2025-11-14 12:36:42.325413: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel 

2372/2375 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.7521 - loss: 1.0430

2025-11-14 12:37:17.538708: E external/local_xla/xla/service/slow_operation_alarm.cc:73] Trying algorithm eng3{k11=0} for conv (f32[13,576,14,14]{3,2,1,0}, u8[0]{0}) custom-call(f32[13,576,14,14]{3,2,1,0}, f32[576,1,3,3]{3,2,1,0}), window={size=3x3 pad=1_1x1_1}, dim_labels=bf01_oi01->bf01, feature_group_count=576, custom_call_target="__cudnn$convForward", backend_config={"operation_queue_id":"0","wait_on_operation_queues":[],"cudnn_conv_backend_config":{"activation_mode":"kNone","conv_result_scale":1,"side_input_scale":0,"leakyrelu_alpha":0},"force_earliest_schedule":false,"reification_cost":[]} is taking a while...
2025-11-14 12:37:17.703031: E external/local_xla/xla/service/slow_operation_alarm.cc:140] The operation took 1.165932497s
Trying algorithm eng3{k11=0} for conv (f32[13,576,14,14]{3,2,1,0}, u8[0]{0}) custom-call(f32[13,576,14,14]{3,2,1,0}, f32[576,1,3,3]{3,2,1,0}), window={size=3x3 pad=1_1x1_1}, dim_labels=bf01_oi01->bf01, feature_group_count=576, custom_call_target="__cudnn

2375/2375 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - accuracy: 0.7522 - loss: 1.0424

2025-11-14 12:37:34.981642: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2025-11-14 12:37:35.149979: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2025-11-14 12:37:35.324602: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2025-11-14 12:37:35.539324: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2025-11-14 12:37:35.721175: E external/local_xla/xla/stream_


Epoch 1: val_accuracy improved from None to 0.93517, saving model to best_model.keras
2375/2375 ━━━━━━━━━━━━━━━━━━━━ 64s 22ms/step - accuracy: 0.8528 - loss: 0.5603 - val_accuracy: 0.9352 - val_loss: 0.2072 - learning_rate: 0.0010
Epoch 2/10
2371/2375 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.9215 - loss: 0.2490
Epoch 2: val_accuracy improved from 0.93517 to 0.94526, saving model to best_model.keras
2375/2375 ━━━━━━━━━━━━━━━━━━━━ 34s 14ms/step - accuracy: 0.9239 - loss: 0.2392 - val_accuracy: 0.9453 - val_loss: 0.1714 - learning_rate: 0.0010
Epoch 3/10
2374/2375 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.9333 - loss: 0.2015
Epoch 3: val_accuracy improved from 0.94526 to 0.94723, saving model to best_model.keras
2375/2375 ━━━━━━━━━━━━━━━━━━━━ 33s 14ms/step - accuracy: 0.9356 - loss: 0.1953 - val_accuracy: 0.9472 - val_loss: 0.1603 - learning_rate: 0.0010
Epoch 4/10
2371/2375 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.9405 - loss: 0.1739
Epoch 4: val_accuracy improved fr

In [12]:
# ==================== PHASE 2: FINE-TUNING ====================
print(f"\n{'='*70}")
print(f" PHASE 2: FINE-TUNING ({FINETUNE_EPOCHS} epochs)")
print(f"{'='*70}")

# Load best model from warmup
print("Loading best model from warmup phase...")
model = tf.keras.models.load_model('best_model.keras')

# Unfreeze base model
base_model = model.get_layer('mobilenetv2_1.00_224')
base_model.trainable = True

# Freeze all except last 40 layers
num_layers = len(base_model.layers)
layers_to_unfreeze = 40
for layer in base_model.layers[:-layers_to_unfreeze]:
    layer.trainable = False

trainable_params = sum([tf.keras.backend.count_params(w) 
                       for w in model.trainable_weights])
print(f"✅ Unfroze last {layers_to_unfreeze} layers of base model")
print(f"   Trainable parameters: {trainable_params:,}")

# Recompile with lower learning rate
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=5e-6),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("\n🚀 Starting fine-tuning...")
history_finetune = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=FINETUNE_EPOCHS,
    callbacks=[checkpoint, reduce_lr, early_stopping],
    class_weight=class_weight,
    verbose=1
)

best_finetune_acc = max(history_finetune.history['val_accuracy'])
print(f"\n✅ Fine-tuning complete!")
print(f"   Best validation accuracy: {best_finetune_acc*100:.2f}%")


 PHASE 2: FINE-TUNING (20 epochs)
Loading best model from warmup phase...
✅ Unfroze last 40 layers of base model
   Trainable parameters: 1,730,214

🚀 Starting fine-tuning...
Epoch 1/20
2375/2375 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 0.6719 - loss: 3.0444
Epoch 1: val_accuracy did not improve from 0.95596
2375/2375 ━━━━━━━━━━━━━━━━━━━━ 61s 21ms/step - accuracy: 0.7760 - loss: 1.5091 - val_accuracy: 0.9294 - val_loss: 0.2325 - learning_rate: 5.0000e-06
Epoch 2/20
2373/2375 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.8782 - loss: 0.3692
Epoch 2: val_accuracy did not improve from 0.95596
2375/2375 ━━━━━━━━━━━━━━━━━━━━ 40s 17ms/step - accuracy: 0.8889 - loss: 0.3336 - val_accuracy: 0.9424 - val_loss: 0.1852 - learning_rate: 5.0000e-06
Epoch 3/20
2373/2375 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 0.9108 - loss: 0.2611
Epoch 3: val_accuracy did not improve from 0.95596

Epoch 3: ReduceLROnPlateau reducing learning rate to 2.499999936844688e-06.
2375/2375 ━━━━━━━━━━━━━━━━━━━

In [13]:
# ==================== EVALUATION ====================
print(f"\n{'='*70}")
print(f" FINAL EVALUATION ON TEST SET")
print(f"{'='*70}")

# Load best model
best_model = tf.keras.models.load_model('best_model.keras')

# Evaluate on test set
print("\n📊 Evaluating on test set...")
test_loss, test_accuracy = best_model.evaluate(test_ds, verbose=0)

print(f"\n🎯 FINAL TEST RESULTS:")
print(f"   Test Accuracy: {test_accuracy*100:.2f}%")
print(f"   Test Loss: {test_loss:.4f}")

if test_accuracy < 0.80:
    print(f"\n⚠️  WARNING: Test accuracy is below 80%!")
    print(f"   Model may not have learned properly")


 FINAL EVALUATION ON TEST SET

📊 Evaluating on test set...


2025-11-14 12:59:16.829009: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2025-11-14 12:59:17.012016: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2025-11-14 12:59:17.251417: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2025-11-14 12:59:17.449262: E external/local_xla/xla/stream_executor/cuda/cuda_timer.cc:86] Delay kernel timed out: measured time has sub-optimal accuracy. There may be a missing warmup execution, please investigate in Nsight Systems.
2025-11-14 12:59:17.541693: E external/local_xla/xla/service


🎯 FINAL TEST RESULTS:
   Test Accuracy: 98.04%
   Test Loss: 0.0758


In [14]:
# ==================== TEST PREDICTIONS ====================
print(f"\n{'='*70}")
print(f" TESTING PREDICTIONS ON SAMPLE IMAGES")
print(f"{'='*70}")

test_cases = [
    ("Apple___Apple_scab", "🍎 Apple Scab"),
    ("Corn_(maize)___healthy", "🌽 Corn Healthy"),
    ("Tomato___Tomato_Yellow_Leaf_Curl_Virus", "🍅 Tomato Virus")
]

all_correct = True

for class_folder, display_name in test_cases:
    class_dir = os.path.join(TEST_DIR, class_folder)
    
    if not os.path.exists(class_dir):
        print(f"\n⚠️  {display_name}: Directory not found")
        continue
    
    image_files = [f for f in os.listdir(class_dir) 
                  if f.lower().endswith(('.jpg', '.jpeg', '.png'))][:2]
    
    if not image_files:
        print(f"\n⚠️  {display_name}: No images found")
        continue
    
    print(f"\n{display_name}:")
    
    for img_file in image_files:
        img_path = os.path.join(class_dir, img_file)
        
        # Load and preprocess
        img = Image.open(img_path).convert("RGB").resize(IMG_SIZE)
        img_array = np.array(img, dtype=np.float32)
        img_array = np.expand_dims(img_array, axis=0)
        
        # Predict
        predictions = best_model.predict(img_array, verbose=0)
        predicted_idx = np.argmax(predictions[0])
        predicted_class = class_names[predicted_idx]
        confidence = predictions[0][predicted_idx]
        # Check correctness
        is_correct = predicted_class == class_folder
        status = "✅" if is_correct else "❌"
        
        if not is_correct:
            all_correct = False
        
        print(f"   {status} {img_file[:35]:35s}")
        print(f"      → {predicted_class[:40]:40s} ({confidence*100:.1f}%)")


 TESTING PREDICTIONS ON SAMPLE IMAGES

🍎 Apple Scab:
   ✅ 0208f4eb-45a4-4399-904e-989ac2c6257
      → Apple___Apple_scab                       (87.3%)
   ✅ 03354abb-aa1c-4f9d-a1ef-9f40505cd53
      → Apple___Apple_scab                       (100.0%)

🌽 Corn Healthy:
   ✅ 03fc887f-e206-4233-8829-ff7070bb344
      → Corn_(maize)___healthy                   (100.0%)
   ✅ 04506233-3040-4d4b-9e8f-acdd0a1e969
      → Corn_(maize)___healthy                   (100.0%)

🍅 Tomato Virus:
   ✅ 00139ae8-d881-4edb-925f-46584b0bd68
      → Tomato___Tomato_Yellow_Leaf_Curl_Virus   (100.0%)
   ✅ 011e0c03-244b-4a90-b087-dececd2828f
      → Tomato___Tomato_Yellow_Leaf_Curl_Virus   (100.0%)


In [17]:
# ==================== SAVE FINAL MODEL ====================
print(f"\n{'='*70}")
print(f" SAVING FINAL MODEL")
print(f"{'='*70}")

best_model.save('plant_disease_model.keras')
print(f"✅ Model saved: plant_disease_model.keras")
print(f"✅ Classes saved: class_names.json")



 SAVING FINAL MODEL
✅ Model saved: plant_disease_model.keras
✅ Classes saved: class_names.json


In [16]:
# ==================== SUMMARY ====================
print(f"\n{'='*70}")
print(f" 🎉 TRAINING COMPLETE!")
print(f"{'='*70}")
print(f"   Final test accuracy: {test_accuracy*100:.2f}%")
print(f"   Model file: plant_disease_model.keras")
print(f"   Classes: {num_classes}")
print(f"   Image size: {IMG_SIZE}")

if all_correct and test_accuracy >= 0.90:
    print(f"\n✅ SUCCESS! Model is working correctly!")
    print(f"   All test predictions were accurate")
    print(f"   Ready for deployment")
elif test_accuracy >= 0.85:
    print(f"\n⚠️  Model trained but some test predictions were wrong")
    print(f"   Review the test predictions above")
else:
    print(f"\n❌ Model may have issues - accuracy too low")
    print(f"   Consider retraining or checking dataset")

print(f"{'='*70}\n")


 🎉 TRAINING COMPLETE!
   Final test accuracy: 98.04%
   Model file: plant_disease_model.keras
   Classes: 38
   Image size: (224, 224)

✅ SUCCESS! Model is working correctly!
   All test predictions were accurate
   Ready for deployment



In [18]:
import tensorflow as tf

# Load the model
model = tf.keras.models.load_model('plant_disease_model.keras')

# Save in legacy format (more compatible)
model.save('plant_disease_model_v2.h5', save_format='h5')

print("✅ Saved in H5 format: plant_disease_model_v2.h5")


✅ Saved in H5 format: plant_disease_model_v2.h5
